# syft-bg Integration Tests

1. **Config & Status basics** — init, inspect config, check status
2. **Misconfigured `email_approve`** — missing `gcp_project_id` → graceful error
3. **Start / Stop / Restart** — verify PID lifecycle
4. **Auto-approve job via API** — `auto_approve_job()`, follow-up auto-approval, list/remove rules
5. **Job repr for auto-approved jobs (DS side)** — verify approval_method shows `auto`
6. **Invalid DO token at startup** — service reports ERROR status

### Prerequisites
- `.env` with `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `APP_CREDENTIALS`, `TOKEN_DS`, `DO_CREDENTIALS`, `DO_TOKEN`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import json
import shutil
import tempfile
import uuid
from pathlib import Path

# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
TOKEN_DS = Path(os.environ["TOKEN_DS"])
DO_CREDENTIALS = Path(os.environ["DO_CREDENTIALS"])
DO_TOKEN = Path(os.environ["DO_TOKEN"])

assert APP_CREDENTIALS.exists(), "App credentials missing"
assert TOKEN_DS.exists(), "DS credentials missing"
assert DO_CREDENTIALS.exists(), "DO credentials missing"
assert DO_TOKEN.exists(), "DO token missing"

In [ ]:
import syft_client as sc

do_client = sc.login_do(email=DO_EMAIL, token_path=str(DO_TOKEN))
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

In [ ]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

try:
    do_client.create_dataset(name="testdata", mock_path="data/mock.txt",
                             private_path="data/private.txt", users=[DS_EMAIL])
except FileExistsError:
    print("Dataset 'testdata' already exists")

---
## Test 1: Config & Status Basics

In [ ]:
import syft_bg
from syft_bg.services.base import ServiceStatus

syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=str(DO_TOKEN),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

In [ ]:
syft_bg.config

In [ ]:
status = syft_bg.status
for name, info in status.service_infos.items():
    assert info.status.value in ("stopped", "unknown"), f"{name} should be stopped, got {info.status}"
print("All services stopped as expected")

---
## Test 2: Misconfigured `email_approve` (no `gcp_project_id`)

In [ ]:
syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=str(DO_TOKEN))

syft_bg.ensure_running(services=["email_approve"])
time.sleep(3)  # wait for subprocess to fail

ea_info = syft_bg.status.service_infos["email_approve"]
assert ea_info.status == ServiceStatus.ERROR, f"Expected ERROR, got {ea_info.status}"
assert ea_info.error and "gcp_project_id" in ea_info.error
print(f"email_approve: {ea_info.status.value}")
print(f"Error confirms missing gcp_project_id")

---
## Test 3: Start / Stop / Restart Services

In [ ]:
syft_bg.reset()
syft_bg.init(do_email=DO_EMAIL, token_path=str(DO_TOKEN),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

SERVICES = ["sync", "notify", "approve"]
syft_bg.ensure_running(services=["sync"], restart=True)
syft_bg.ensure_running(services=["notify", "approve"], restart=True)

In [ ]:
status = syft_bg.status
initial_pids = {}
for svc in SERVICES:
    info = status.service_infos[svc]
    assert info.status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), f"{svc}: {info.status}"
    initial_pids[svc] = info.pid
    print(f"{svc}: PID {info.pid}")
print(f"All {len(SERVICES)} services running")

In [ ]:
# Restart approve — PID should change
syft_bg.restart("approve")
new_pid = syft_bg.status.service_infos["approve"].pid
assert new_pid != initial_pids["approve"], f"PID didn't change: {new_pid}"
print(f"approve restarted: {initial_pids['approve']} -> {new_pid}")

In [ ]:
# Stop approve, others keep running
syft_bg.stop("approve")
s = syft_bg.status
assert s.service_infos["approve"].status == ServiceStatus.STOPPED
for svc in ["sync", "notify"]:
    assert s.service_infos[svc].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING)
print("approve stopped, sync/notify still running")

In [ ]:
# Start approve again
syft_bg.start("approve")
assert syft_bg.status.service_infos["approve"].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING)
print("Start/Stop/Restart test passed")

---
## Test 4: Auto-Approve Job via API

In [ ]:
def create_parameterized_job(main_file, params):
    d = Path(tempfile.mkdtemp(prefix="syftbg_test_"))
    (d / main_file).write_text(
        'import os, json\n'
        'with open("params.json") as f: params = json.load(f)\n'
        'os.makedirs("outputs", exist_ok=True)\n'
        'with open("outputs/result.json", "w") as f:\n'
        '    json.dump({"params": params, "status": "done"}, f)\n'
        'print(f"Result: {params}")\n'
    )
    (d / "params.json").write_text(json.dumps(params))
    return d

In [ ]:
syft_bg.ensure_running(services=["sync"], restart=True)
syft_bg.ensure_running(services=["notify", "approve"], restart=True)
time.sleep(10)  # wait for services to be fully up
syft_bg.status

In [ ]:
syft_bg.status

In [ ]:
syft_bg.status

In [ ]:
JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
ds_client.submit_python_job(user=DO_EMAIL, code_path=str(create_parameterized_job("main.py", {"run": 1})),
                            job_name=JOB_NAME, entrypoint="main.py", force_submission=True)
print(f"Submitted: {JOB_NAME}")

In [ ]:
# Wait for job to appear on DO side (sync service keeps local state updated)
pending_job = None
for _ in range(24):
    pending_job = next((j for j in do_client.job_client.jobs if j.name == JOB_NAME), None)
    if pending_job:
        break
    time.sleep(5)

assert pending_job, f"DO never saw job {JOB_NAME}"
print(f"DO sees job: {JOB_NAME} ({pending_job.status})")

In [ ]:
result = syft_bg.auto_approve_job(pending_job, file_paths=["params.json"])
assert result.success, f"auto_approve_job failed: {result.error}"
print(result)

In [ ]:
# Verify rule
rules = syft_bg.list_auto_approvals()
assert JOB_NAME in rules
rule = rules[JOB_NAME]
assert {e.relative_path for e in rule.file_contents} == {"main.py"}
assert "params.json" in rule.file_paths
assert DS_EMAIL in rule.peers
assert JOB_NAME in syft_bg.status.auto_approvals
print(f"Rule '{JOB_NAME}' created and visible in status")

In [ ]:
# Wait for job to complete
for _ in range(24):
    job = next((j for j in do_client.job_client.jobs if j.name == JOB_NAME), None)
    if job and job.status == "done":
        break
    time.sleep(5)

assert job and job.status == "done", f"Job not done: {job.status if job else 'missing'}"
print(f"Job {JOB_NAME}: {job.status}")

In [ ]:
# DS fetches result
ds_job = next((j for j in ds_client.job_client.jobs if j.name == JOB_NAME), None)
assert ds_job and ds_job.status == "done"
result_data = json.loads(ds_job.output_paths[0].read_text())
assert result_data["params"]["run"] == 1
print(f"DS result: {result_data}")

In [ ]:
# Follow-up job — should auto-approve
FOLLOWUP_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"
ds_client.submit_python_job(user=DO_EMAIL, code_path=str(create_parameterized_job("main.py", {"run": 2})),
                            job_name=FOLLOWUP_NAME, entrypoint="main.py", force_submission=True)
print(f"Submitted follow-up: {FOLLOWUP_NAME}")

In [ ]:
for _ in range(24):
    job2 = next((j for j in do_client.job_client.jobs if j.name == FOLLOWUP_NAME), None)
    if job2 and job2.status == "done":
        break
    time.sleep(5)

assert job2 and job2.status == "done", f"Follow-up not done: {job2.status if job2 else 'missing'}"
print(f"Follow-up {FOLLOWUP_NAME}: {job2.status} (auto-approved!)")

In [ ]:
ds_job2 = next((j for j in ds_client.job_client.jobs if j.name == FOLLOWUP_NAME), None)
assert ds_job2 and ds_job2.status == "done"
result_data2 = json.loads(ds_job2.output_paths[0].read_text())
assert result_data2["params"]["run"] == 2
print(f"DS follow-up result: {result_data2}")

---
## Test 5: Job Repr for Auto-Approved Jobs (DS Side)

In [ ]:
for label, j in [("job1", ds_job), ("job2", ds_job2)]:
    assert "[approved: auto]" in str(j), f"{label} str missing auto"
    assert "approval_method='auto'" in repr(j), f"{label} repr missing auto"
    print(f"{label}: {str(j)}")

assert "auto" in ds_job._repr_html_()
jobs_str = str(ds_client.jobs)
for name in [JOB_NAME, FOLLOWUP_NAME]:
    lines = [l for l in jobs_str.splitlines() if name in l]
    assert lines and "auto" in lines[0], f"Missing 'auto' for {name}"
assert "auto" in ds_client.jobs._repr_html_()
print("All repr tests passed")

In [ ]:
# Remove rule
r = syft_bg.remove_auto_approve(JOB_NAME)
assert r.success
assert JOB_NAME not in syft_bg.list_auto_approvals()
print(f"Rule '{JOB_NAME}' removed")

---
## Test 6: Invalid DO Token at Startup

In [ ]:
syft_bg.reset()
bad_token = Path(tempfile.mktemp(suffix=".json"))
bad_token.write_text(json.dumps({"invalid": True}))

syft_bg.init(do_email=DO_EMAIL, token_path=str(bad_token),
             settings={"email_approve": {"gcp_project_id": GCP_PROJECT_ID}})

syft_bg.ensure_running(services=["sync"])
time.sleep(3)  # wait for subprocess to fail

sync_info = syft_bg.status.service_infos["sync"]
assert sync_info.status == ServiceStatus.ERROR, f"Expected ERROR, got {sync_info.status}"
assert sync_info.error
print(f"sync: {sync_info.status.value}")
print(f"Error: {sync_info.error[:200]}...")

bad_token.unlink(missing_ok=True)
syft_bg.stop()
print("\nTest 6 passed")

---
## Debugging

In [ ]:
for svc in ("sync", "notify", "approve"):
    print(f"\n{'='*20} {svc} {'='*20}")
    syft_bg.logs(svc, n=20)

In [ ]:
syft_bg.status

---
## Cleanup

In [ ]:
syft_bg.stop()